## Model Selection and Prompt Optimization

In this notebook, we will demonstrate how the NVIDIA NeMo Agent toolkit (NAT) <a href="https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/evaluate.html">evaluators</a> can be used to develop robust model selection and prompt optimization workflows.

**Goal**: show how the parameter optimizer can be used to compare models and prompts.

## Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

### API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [1]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

## Installing NeMo Agent Toolkit

The recommended way to install NAT is through `pip` or `uv pip`.

First, we will install `uv` which offers parallel downloads and faster dependency resolution.

In [2]:
%%bash
pip install uv

NeMo Agent toolkit can be installed through the PyPI `nvidia-nat` package.

There are several optional subpackages available for NAT. For this example, we will rely on two subpackages:
* The `langchain` subpackage contains useful components for integrating and running within [LangChain](https://python.langchain.com/docs/introduction/).
* The `llama-index` subpackage contains useful components for integrating and running within [LlamaIndex](https://developers.llamaindex.ai/python/framework/).

In [ ]:
%%bash
uv pip install "nvidia-nat[langchain,phoenix,profiling]"

## 1) Simple Chat Completions Accuracy Comparison (with LLM as a judge)

Create a basic chat completions workflow (uses LangChain chat completions on backend)

In [3]:
!nat workflow create eval_workflow --description "A simple ReAct agent that can use tools"

Installing workflow 'eval_workflow'...
Workflow 'eval_workflow' installed successfully.
Workflow 'eval_workflow' created successfully in '/Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/notebooks/eval_workflow'.


Let's look at the default configuration of this agent and confirm the agent type, llms, tool calls, and functions...

In [4]:
%%writefile ./eval_workflow/configs/config_a.yml
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.7
    max_tokens: 1024

workflow:
  _type: chat_completion  # Use the type directly
  system_prompt: |
    You are a helpful AI assistant. Provide clear, accurate, and helpful 
    responses to user queries. Be concise and informative.
  llm_name: nim_llm

Writing ./eval_workflow/configs/config_a.yml


Now let's run this workflow for a simple Q&A example...

In [5]:
!nat run --config_file eval_workflow/configs/config_a.yml --input "Suggest a single name for my new dog"

2025-10-16 10:25:29 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'eval_workflow/configs/config_a.yml'
2025-10-16 10:25:29 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-10-16 10:25:35 - WARNING  - nat.builder.intermediate_step_manager:94 - Step id 17d43440-7f7e-401d-b166-12792defd17f not found in outstanding start steps
2025-10-16 10:25:35 - INFO     - nat.front_ends.console.console_front_

## 2) observing a workflow with Phoenix

> **Note:** _This portion of the example will only work when the notebook is run locally. It may not work through Google Colab and other online notebook environments._

Phoenix is an open-source observability platform designed for monitoring, debugging, and improving LLM applications and AI agents. It provides a web-based interface for visualizing and analyzing traces from LLM applications, agent workflows, and ML pipelines. Phoenix automatically captures key metrics such as latency, token usage, and costs, and displays the inputs and outputs at each step, making it invaluable for debugging complex agent behaviors and identifying performance bottlenecks in AI workflows.

### Updating the Workflow Configuration

We will need to update the workflow configuration file to support telemetry tracing with Phoenix.

To do this, we will first copy the original configuration:

In [6]:
!cp eval_workflow/configs/config_a.yml eval_workflow/configs/config_b.yml

Then we will append necessary configuration components to the `phoenix_config.yml` file:

In [7]:
%%writefile -a eval_workflow/configs/config_b.yml

general:
  telemetry:
    logging:
      console:
        _type: console
        level: WARN
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: my_react_agent


Appending to eval_workflow/configs/config_b.yml


### Start Phoenix Server

First, we will install Phoenix:

In [8]:
!uv pip install arize-phoenix

Using Python 3.12.11 environment at: /Users/bbednarski/.venvs/unew_312
Resolved 79 packages in 241ms                                        
Installed 6 packages in 69ms                                
 + arize-phoenix==12.6.1
 + arize-phoenix-evals==2.5.0
 + email-validator==2.3.0
 + pystache==0.6.8
 + sqlean-py==3.50.4.4
 + strawberry-graphql==0.270.1


Then, we will ensure the service is publicly accessible:

In [9]:
%env PHOENIX_HOST=0.0.0.0

env: PHOENIX_HOST=0.0.0.0


Finally, we will start the server:

In [10]:
%%bash --bg
# phoenix will run on port 6006
phoenix serve

### Running the Workflow

Instead of the original workflow configuration, we will run with the updated `phoenix_config.yml` file:

In [11]:
!nat run --config_file eval_workflow/configs/config_b.yml --input "Suggest a single name for my new cat."

2025-10-16 10:26:26 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'eval_workflow/configs/config_b.yml'
2025-10-16 10:26:30 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.

Configuration Summary:
--------------------
Workflow Type: chat_completion
Number of Functions: 0
Number of Function Groups: 0
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-10-16 10:27:28 - WARNING  - nat.builder.intermediate_step_manager:94 - Step id 8440b5be-2f1d-4b42-9f7b-bfd43e3599b4 not found in outstanding start steps
2025-10-16 10:27:28 - WARNING  - nat.observability.exporter.span_expor

Results of this latest workflow execution should now be visible on your locally hosted Phoenix dashboard under the respective project name and trace.

## 3) Head-to-head comparison of multiple LLMs using eval

In this next section, we are going to update the workflow configuration for evaluation and profiling.

Step by step instructions can be found in [4_observability_evaluation_and_profiling.ipynb](./4_observability_evaluation_and_profiling.ipynb). An end to end example of using the Optimizer can be viewed in the [email_phishing_analyzer](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/develop/examples/evaluation_and_profiling/email_phishing_analyzer/src/nat_email_phishing_analyzer/configs/config_optimizer.yml)

The profiler instruments and measures your workflow's performance, while evaluators judge the quality of the outputs. They're separate concepts, so they belong in different sections of the config!

In this next step we will combine the eval and profile configuration into a single config for brevity.

In [12]:
%%writefile eval_workflow/configs/config_c.yml
llms:
  nim_llm_8b:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.0

  nim_llm_70b:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0

  # Judge LLM for accuracy evaluation
  nim_judge_llm:
    _type: nim
    model_name: meta/llama-3.1-405b-instruct
    temperature: 0.0
    max_tokens: 8  # RAGAS accuracy only needs a score (0-1)

workflow:
  _type: chat_completion
  system_prompt: |
    You are a helpful AI assistant. Provide clear, accurate, and helpful 
    responses to user queries. Be concise and informative.
  llm_name: nim_llm_8b

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: eval_workflow

eval:
  general:
    output_dir: ./eval_workflow/eval_output
    verbose: true
    dataset:
        _type: json
        file_path: ./eval_workflow/data/eval_data.json

  evaluators:
    answer_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_judge_llm
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end

  profiler:
      token_uniqueness_forecast: true
      workflow_runtime_forecast: true
      compute_llm_metrics: true
      csv_exclude_io_text: true
      prompt_caching_prefixes:
        enable: true
        min_frequency: 0.1
      bottleneck_analysis:
        enable_nested_stack: true
      concurrency_spike_analysis:
        enable: true
        spike_threshold: 7

# optimizer:
#   output_path: ./eval_workflow/eval_output/optimizer/
#   reps_per_param_set: 1
#   eval_metrics:
#     accuracy:
#       evaluator_name: answer_accuracy  # References the evaluator above
#       direction: maximize
#       weight: 0.5  # Adjust weight based on importance
    
#     token_efficiency:
#       evaluator_name: token_efficiency
#       direction: minimize
#       weight: 0.3
    
#     latency:
#       evaluator_name: llm_latency
#       direction: minimize
#       weight: 0.2


#   numeric:
#     enabled: true
#     n_trials: 3

#     search_space:
#       workflow:
#         llm_name:
#           _type: categorical
#           choices: [nim_llm_8b, nim_llm_70b]  # Models to compare

#   prompt:
#     enabled: false  # Disable for pure model comparison

Writing eval_workflow/configs/config_c.yml


Adding the test dataset...

In [13]:
%%writefile eval_workflow/data/eval_data.json
[
    {
        "id": "1",
        "question": "What is 15% of 847?",
        "answer": "The answer is: 127.05"
    }
]

Writing eval_workflow/data/eval_data.json


Evaluate the nim_llm_8b model...

In [ ]:
%%bash
# Evaluate with 8B model
nat eval --config_file eval_workflow/configs/config_c.yml \
    --override workflow.llm_name nim_llm_8b \
    --override eval.general.output_dir ./eval_workflow/eval_output/8b

2025-10-16 10:27:55 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: eval_workflow/configs/config_c.yml
2025-10-16 10:27:59 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for workflow.llm_name with value: nim_llm_8b with type <class 'str'>)
2025-10-16 10:27:59 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for eval.general.output_dir with value: ./eval_workflow/eval_output/8b with type <class 'str'>)
2025-10-16 10:27:59 - INFO     - nat.cli.cli_utils.config_override:211 - 

Configuration after overrides:

eval:
  evaluators:
    answer_accuracy:
      _type: ragas
      llm_name: nim_judge_llm
      metric: AnswerAccuracy
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end
  general:
    dataset:
      _type: json
      file_path: ./eval_workflow/data/eval_data.json
    output_dir: ./eval_workflow/eval_output/8b
    verbose: true
  profiler:


2025-10-16 10:27:59 - INFO     - phoenix.config:1521 - 📋 Ensuring phoenix working directory: /Users/bbednarski/.phoenix
2025-10-16 10:27:59 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_9e632d02-cbc5-496b-bf57-c657f5d99bd2 initialized
2025-10-16 10:28:01 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.


Running workflow:   0%|          | 0/1 [00:00<?, ?it/s]

2025-10-16 10:28:05 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-16 10:28:07 - WARNING  - nat.builder.intermediate_step_manager:94 - Step id ed205529-2d6c-4e2b-929d-7311049df547 not found in outstanding start steps
2025-10-16 10:28:07 - INFO     - nat.observability.exporter.base_exporter:283 - Event stream completed. No more events will arrive.
2025-10-16 10:28:07 - WARNING  - nat.observability.exporter.span_exporter:300 - Not all spans were closed. Remaining: {'ed205529-2d6c-4e2b-929d-7311049df547': Span(name='<workflow>', context=SpanContext(trace_id=153779088216035442295189067251571722489, span_id=13190974102474278149), parent=None, start_time=1760635685634307072, end_time=None, attributes={'nat.event_type': 'WORKFLOW_START', 'nat.function.id': 'root', 'nat.function.name': 'root', 'nat.subspan.name': '<workflow>', 'nat.event_timestamp': 1760635685.6343071, 'nat.framework': 'unknown', 'nat.conversation.id': 'unknown', 'nat.workflow.run_id':

Evaluating Avg LLM Latency:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating Ragas nv_accuracy: 100%|██████████| 1/1 [00:01<00:00,  1.64s/it]


2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:426 - Waiting for export tasks from 1 local exporters (timeout: 60s)
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:431 - Export tasks completed for exporter: phoenix
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:435 - All local export task waiting completed


2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:250 - Profiler is not enabled. Skipping profiling.
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:335 - Workflow output written to eval_workflow/eval_output/8b/workflow_output.json
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/8b/llm_latency_output.json
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/8b/token_efficiency_output.json
2025-10-16 10:28:09 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/8b/answer_accuracy_output.json


In [15]:
%%bash
# Evaluate with 70B model
nat eval --config_file eval_workflow/configs/config_c.yml \
    --override workflow.llm_name nim_llm_70b \
    --override eval.general.output_dir ./eval_workflow/eval_output/70b

2025-10-16 10:28:52 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: eval_workflow/configs/config_c.yml
2025-10-16 10:28:56 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for workflow.llm_name with value: nim_llm_70b with type <class 'str'>)
2025-10-16 10:28:56 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for eval.general.output_dir with value: ./eval_workflow/eval_output/70b with type <class 'str'>)
2025-10-16 10:28:56 - INFO     - nat.cli.cli_utils.config_override:211 - 

Configuration after overrides:

eval:
  evaluators:
    answer_accuracy:
      _type: ragas
      llm_name: nim_judge_llm
      metric: AnswerAccuracy
    llm_latency:
      _type: avg_llm_latency
    token_efficiency:
      _type: avg_tokens_per_llm_end
  general:
    dataset:
      _type: json
      file_path: ./eval_workflow/data/eval_data.json
    output_dir: ./eval_workflow/eval_output/70b
    verbose: true
  profile

2025-10-16 10:28:56 - INFO     - phoenix.config:1521 - 📋 Ensuring phoenix working directory: /Users/bbednarski/.phoenix
2025-10-16 10:28:56 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_7e208336-ab29-40b7-b9de-d3d2b468b54f initialized
2025-10-16 10:28:58 - WARNING  - nat.profiler.utils:137 - Discovered frameworks: {<LLMFrameworkEnum.LANGCHAIN: 'langchain'>} in function register_chat_completion by inspecting source. It is recommended and more reliable to instead add the used LLMFrameworkEnum types in the framework_wrappers argument when calling @register_function.


Running workflow:   0%|          | 0/1 [00:00<?, ?it/s]

2025-10-16 10:28:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-16 10:29:01 - WARNING  - nat.builder.intermediate_step_manager:94 - Step id b50db338-3d30-42eb-840b-3065efa390ba not found in outstanding start steps
2025-10-16 10:29:01 - INFO     - nat.observability.exporter.base_exporter:283 - Event stream completed. No more events will arrive.
2025-10-16 10:29:01 - WARNING  - nat.observability.exporter.span_exporter:300 - Not all spans were closed. Remaining: {'b50db338-3d30-42eb-840b-3065efa390ba': Span(name='<workflow>', context=SpanContext(trace_id=191161744363109591381721066798403519327, span_id=18287347837423010962), parent=None, start_time=1760635739443130880, end_time=None, attributes={'nat.event_type': 'WORKFLOW_START', 'nat.function.id': 'root', 'nat.function.name': 'root', 'nat.subspan.name': '<workflow>', 'nat.event_timestamp': 1760635739.443131, 'nat.framework': 'unknown', 'nat.conversation.id': 'unknown', 'nat.workflow.run_id': 

Evaluating Avg LLM Latency:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating Ragas nv_accuracy: 100%|██████████| 1/1 [00:01<00:00,  1.27s/it]


2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:426 - Waiting for export tasks from 1 local exporters (timeout: 60s)
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:431 - Export tasks completed for exporter: phoenix
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:435 - All local export task waiting completed


2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:250 - Profiler is not enabled. Skipping profiling.
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:335 - Workflow output written to eval_workflow/eval_output/70b/workflow_output.json
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/70b/llm_latency_output.json
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/70b/token_efficiency_output.json
2025-10-16 10:29:02 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/70b/answer_accuracy_output.json


Next lets compare the performance of each model...

In [19]:
import json
from pathlib import Path

# Define the test names to compare
test_names = ['8b', '70b']

print("Model Performance Comparison")
print("=" * 80)

for test_name in test_names:
    print(f"\n{test_name.upper()} Model:")
    print("-" * 40)
    
    # Load answer accuracy results
    accuracy_path = Path(f"./eval_workflow/eval_output/{test_name}/answer_accuracy_output.json")
    if accuracy_path.exists():
        with open(accuracy_path, 'r') as f:
            accuracy_data = json.load(f)
            avg_accuracy = accuracy_data.get('average_score', 'N/A')
            print(f"  Average Accuracy Score: {avg_accuracy}")
    else:
        print(f"  Average Accuracy Score: N/A (file not found)")
    
    # Load LLM latency results
    latency_path = Path(f"./eval_workflow/eval_output/{test_name}/llm_latency_output.json")
    if latency_path.exists():
        with open(latency_path, 'r') as f:
            latency_data = json.load(f)
            avg_latency = latency_data.get('average_score', 'N/A')
            print(f"  Average LLM Latency: {avg_latency} sec")
    else:
        print(f"  Average LLM Latency: N/A (file not found)")
    
    # Load token efficiency results
    token_path = Path(f"./eval_workflow/eval_output/{test_name}/token_efficiency_output.json")
    if token_path.exists():
        with open(token_path, 'r') as f:
            token_data = json.load(f)
            avg_tokens = token_data.get('average_score', 'N/A')
            print(f"  Average Tokens per LLM_END: {avg_tokens}")
    else:
        print(f"  Average Tokens per LLM_END: N/A (file not found)")

print("\n" + "=" * 80)


Model Performance Comparison

8B Model:
----------------------------------------
  Average Accuracy Score: 0.25
  Average LLM Latency: 2.33 sec
  Average Tokens per LLM_END: 94.0

70B Model:
----------------------------------------
  Average Accuracy Score: 1.0
  Average LLM Latency: 2.16 sec
  Average Tokens per LLM_END: 94.0



## Understanding Evaluation Outputs

This evaluation will have generated two artifacts for analysis at the `output_dir` specified in `config_c.yaml`:
- **trajectory_accuracy_output.json**
- **workflow_output.json**

### Interpreting trajectory_accuracy_output.json

The `trajectory_accuracy_output.json` file contains the results of agent trajectory evaluation.

#### Top-level fields:
- **average_score** - Mean trajectory accuracy score across all evaluated examples (0.0 to 1.0)
- **eval_output_items** - Array of individual evaluation results for each test case

#### Per-item fields:
- **id** - Unique identifier for the test case
- **score** - Trajectory accuracy score for this specific example (0.0 to 1.0)
- **reasoning** - Evaluation reasoning, either:
  - String containing error message if evaluation failed
  - Object with:
    - **reasoning** - LLM judge's explanation of the score
    - **trajectory** - Array of [AgentAction, Output] pairs showing the agent's execution path

The trajectory accuracy evaluator assesses whether the agent used appropriate tools, followed a logical sequence of steps, and efficiently reached the correct answer.

### Interpreting workflow_output.json

The `workflow_output.json` file contains the raw execution results from running the workflow on each test case.

#### Top-level fields:
- **output_items** - Array of workflow execution results for each test case in the dataset

#### Per-item fields:
- **id** - Unique identifier matching the test case ID
- **input_obj** - The input question or prompt sent to the workflow
- **output_obj** - The final answer generated by the workflow
- **trajectory** - Detailed execution trace containing:
  - **event_type** - Type of event (e.g., `LLM_START`, `LLM_END`, `TOOL_START`, `TOOL_END`, `SPAN_START`, `SPAN_END`)
  - **event_timestamp** - Unix timestamp of when the event occurred
  - **metadata** - Event-specific data including:
    - Tool names and inputs
    - LLM prompts and responses
    - Token counts (`prompt_tokens`, `completion_tokens`)
    - Model names
    - Function names
    - Error information

The workflow output provides complete observability into each execution, enabling detailed analysis of agent behavior, performance profiling, and debugging.

## 4) integrating a tool-calling agent

Next, we are going to integrate and advanced agent, the [Alert Triage Agent](https://github.com/NVIDIA/NeMo-Agent-Toolkit/tree/develop/examples/advanced_agents/alert_triage_agent), which uses tool calling to automate the triage of server-monitoring alerts. It demonstrates how to build an intelligent troubleshooting workflow using NeMo Agent toolkit and LangGraph.

The Alert Triage Agent is an advanced example that demonstrates:
- **Multi-tool orchestration** - Dynamically selects and uses diagnostic tools
- **Structured report generation** - Creates comprehensive analysis reports
- **Root cause categorization** - Classifies alerts into predefined categories
- **Offline evaluation mode** - Test with synthetic data before live deployment

First, we'll install the alert triage agent package:

In [ ]:
%%bash
# The -e flag installs in "editable" mode, meaning you can modify the source files
# and see changes immediately without reinstalling
uv pip install -e ../../examples/advanced_agents/alert_triage_agent

# This registers all the custom functions (hardware_check, maintenance_check, etc.)
# so they're available to use in our local configuration

Using Python 3.12.11 environment at: /Users/bbednarski/.venvs/unew_312
Resolved 157 packages in 4.13s
   Building nat-alert-triage-agent @ file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent
      Built nat-alert-triage-agent @ file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent
Prepared 1 package in 692ms
Uninstalled 1 package in 8ms
Installed 1 package in 12ms
 - nat-alert-triage-agent==1.3.0rc4.dev10+gd91fd6cf (from file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent)
 + nat-alert-triage-agent==1.3.0rc4.dev26+ge0835661 (from file:///Users/bbednarski/Projects/nat-getting-started-fork/NeMo-Agent-Toolkit/examples/advanced_agents/alert_triage_agent)


### Working Locally with the Alert Triage Agent

The editable installation (`-e`) means:
- The source files remain at `../../examples/advanced_agents/alert_triage_agent`
- You can edit the Python files directly and changes take effect immediately
- All custom functions are registered and available for use
- Your local configuration file (created below) references those functions

**Alternative: Fully Local Workflow**
If you want a completely local workflow without any installation, you could:
1. Create a new workflow: `nat workflow create alert_triage_local`
2. Copy the function implementations to your local `src/` directory
3. Register them in your local `register.py`

For this notebook, we'll use the editable install approach, which gives you the best of both worlds.

### Configuring the Alert Triage Agent

The Alert Triage Agent requires several components:

1. **Diagnostic Tools** - Hardware checks, network connectivity, performance monitoring, telemetry analysis
2. **Sub-agents** - Telemetry metrics analysis agent that coordinates multiple telemetry tools
3. **Categorizer** - Classifies root causes into predefined categories
4. **Maintenance Check** - Filters out alerts during maintenance windows

We'll create a **local configuration file** and run in **offline mode** using synthetic data:


In [ ]:
%%writefile ./eval_workflow/configs/alert_triage_config.yml
functions:
  hardware_check:
    _type: hardware_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  host_performance_check:
    _type: host_performance_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  monitoring_process_check:
    _type: monitoring_process_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  network_connectivity_check:
    _type: network_connectivity_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_host_heartbeat_check:
    _type: telemetry_metrics_host_heartbeat_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_host_performance_check:
    _type: telemetry_metrics_host_performance_check
    llm_name: tool_reasoning_llm
    offline_mode: true
  telemetry_metrics_analysis_agent:
    _type: telemetry_metrics_analysis_agent
    tool_names:
      - telemetry_metrics_host_heartbeat_check
      - telemetry_metrics_host_performance_check
    llm_name: agent_llm
  maintenance_check:
    _type: maintenance_check
    llm_name: agent_llm
    static_data_path: ../../examples/advanced_agents/alert_triage_agent/data/maintenance_static_dataset.csv
  categorizer:
    _type: categorizer
    llm_name: agent_llm

workflow:
  _type: alert_triage_agent
  tool_names:
    - hardware_check
    - host_performance_check
    - monitoring_process_check
    - network_connectivity_check
    - telemetry_metrics_analysis_agent
  llm_name: agent_llm
  offline_mode: true
  offline_data_path: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.csv
  benign_fallback_data_path: ../../examples/advanced_agents/alert_triage_agent/data/benign_fallback_offline_data.json

llms:
  agent_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    temperature: 0.2
    max_tokens: 2048

  tool_reasoning_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.2
    max_tokens: 2048

  nim_rag_eval_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    max_tokens: 8

eval:
  general:
    output_dir: ./eval_workflow/alert_triage_output/
    dataset:
      _type: json
      file_path: ../../examples/advanced_agents/alert_triage_agent/data/offline_data.json
  evaluators:
    classification_accuracy:
      _type: classification_accuracy
    rag_accuracy:
      _type: ragas
      metric: AnswerAccuracy
      llm_name: nim_rag_eval_llm


### Running a Single Alert Example

Let's test the Alert Triage Agent with a single alert. This alert is an "InstanceDown" alert that, according to the offline dataset, is actually a false positive (the system is healthy).


In [ ]:
!nat run --config_file eval_workflow/configs/alert_triage_config.yml \
  --input '{"alert_id": 0, "alert_name": "InstanceDown", "host_id": "test-instance-0.example.com", "severity": "critical", "description": "Instance test-instance-0.example.com is not available for scrapping for the last 5m. Please check: - instance is up and running; - monitoring service is in place and running; - network connectivity is ok", "summary": "Instance test-instance-0.example.com is down", "timestamp": "2025-04-28T05:00:00.000000"}'


### Evaluating the Alert Triage Agent

Now let's run a full evaluation on the Alert Triage Agent using the complete offline dataset. This dataset contains seven alerts with different root causes:

- **False positives** - System appears healthy despite alert
- **Hardware issues** - Hardware failures or degradation  
- **Software issues** - Malfunctioning monitoring services
- **Maintenance** - Scheduled maintenance windows
- **Repetitive behavior** - Benign recurring patterns

The evaluation will measure:
1. **Classification Accuracy** - How well the agent categorizes root causes
2. **Answer Accuracy** - How well the generated reports match expected outcomes (using RAGAS)


In [ ]:
%%bash
nat eval --config_file ./eval_workflow/configs/alert_triage_config.yml


### Understanding Alert Triage Evaluation Results

The evaluation generates several output files in the `alert_triage_output` directory:

1. **classification_accuracy_output.json** - Root cause classification metrics
   - Shows accuracy, precision, recall, and F1 scores for each category
   - Contains confusion matrix for detailed analysis
   
2. **rag_accuracy_output.json** - Answer quality metrics
   - Measures how well generated reports match expected outcomes
   - Uses LLM-as-a-judge to evaluate report quality

3. **workflow_output.json** - Complete execution traces
   - Contains full agent trajectories with tool calls
   - Includes generated reports for each alert
   - Shows token usage and performance metrics

Let's examine the classification accuracy results:


In [ ]:
import json
from pathlib import Path

# Read classification accuracy results
classification_results_path = Path("./eval_workflow/alert_triage_output/classification_accuracy_output.json")
if classification_results_path.exists():
    with open(classification_results_path) as f:
        classification_results = json.load(f)
    
    print("Classification Accuracy Results:")
    print("=" * 60)
    print(f"Overall Accuracy: {classification_results.get('average_score', 'N/A'):.2%}")
    print("\nPer-Category Metrics:")
    
    eval_items = classification_results.get('eval_output_items', [])
    if eval_items:
        # Aggregate metrics by category
        category_scores = {}
        for item in eval_items:
            label = item.get('label', 'unknown')
            score = item.get('score', 0)
            if label not in category_scores:
                category_scores[label] = []
            category_scores[label].append(score)
        
        for category, scores in sorted(category_scores.items()):
            avg_score = sum(scores) / len(scores)
            print(f"  {category}: {avg_score:.2%} ({len(scores)} samples)")
else:
    print("Classification results not yet available. Run the evaluation first.")


### Examining a Sample Triage Report

Let's look at one of the generated triage reports to understand the agent's reasoning:


In [ ]:
from IPython.display import Markdown

# Read workflow output to get a sample report
workflow_results_path = Path("./eval_workflow/alert_triage_output/workflow_output.json")
if workflow_results_path.exists():
    with open(workflow_results_path) as f:
        workflow_results = json.load(f)
    
    output_items = workflow_results.get('output_items', [])
    if output_items:
        # Display the first report
        first_item = output_items[0]
        alert_input = json.loads(first_item.get('input_obj', '{}'))
        report = first_item.get('output_obj', 'No report generated')
        
        print(f"Alert: {alert_input.get('alert_name')} on {alert_input.get('host_id')}")
        print(f"Severity: {alert_input.get('severity')}")
        print("\nGenerated Triage Report:")
        print("-" * 80)
        display(Markdown(report[:2000] + "..." if len(report) > 2000 else report))
else:
    print("Workflow results not yet available. Run the evaluation first.")


### Key Takeaways - Alert Triage Agent

The Alert Triage Agent demonstrates several advanced NeMo Agent toolkit capabilities:

1. **Multi-tool orchestration** - The agent dynamically selects from five diagnostic tools based on alert context
2. **Hierarchical agents** - The telemetry metrics analysis agent is itself composed of multiple specialized tools
3. **Structured outputs** - Generates comprehensive markdown reports with clear sections and recommendations
4. **Root cause classification** - Automatically categorizes issues into predefined categories for faster response
5. **Offline testing** - Supports evaluation with synthetic data before live deployment

### Potential Extensions

To further develop this workflow, consider:

- **Model comparison** - Compare different LLMs (such as 8B compared to 70B compared to 405B models) on classification accuracy
- **Prompt optimization** - Experiment with different system prompts for the triage agent
- **Tool selection strategies** - Compare which combinations of tools provide best accuracy
- **Live deployment** - Connect to real monitoring systems and infrastructure
- **Custom evaluators** - Add domain-specific metrics for your alert types

For more details, see the [Alert Triage Agent documentation](https://github.com/NVIDIA/NeMo-Agent-Toolkit/tree/develop/examples/advanced_agents/alert_triage_agent).


In [55]:
%%writefile eval_workflow/configs/config_d.yml
functions:
  current_datetime:
    _type: current_datetime

llms:
  nim_llm_8b:
    _type: nim
    model_name: meta/llama-3.1-8b-instruct
    temperature: 0.0

  nim_llm_70b:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0
  
  nim_llm_405b:
    _type: nim
    model_name: meta/llama-3.1-405b-instruct
    temperature: 0.0

workflow:
  _type: react_agent
  tool_names: [current_datetime]
  llm_name: nim_llm
  verbose: true
  max_retries: 3

general:
  telemetry:
    logging:
      console:
        _type: console
        level: INFO
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: eval_workflow

eval:
  general:
    output_dir: ./eval_workflow/eval_output
    verbose: true
    dataset:
        _type: json
        file_path: ./eval_workflow/data/eval_data.json

  evaluators:
    trajectory_accuracy:
      _type: trajectory
      llm_name: nim_llm

  profiler:
      token_uniqueness_forecast: true
      workflow_runtime_forecast: true
      compute_llm_metrics: true
      csv_exclude_io_text: true
      prompt_caching_prefixes:
        enable: true
        min_frequency: 0.1
      bottleneck_analysis:
        enable_nested_stack: true
      concurrency_spike_analysis:
        enable: true
        spike_threshold: 7


Overwriting eval_workflow/configs/config_d.yml


In [60]:
%%bash
# 8B model
nat eval --config_file ./eval_workflow/configs/config_d.yml \
  --override workflow.llm_name nim_llm_8b \
  --override eval.evaluators.trajectory_accuracy.llm_name nim_llm_8b \
  --override eval.general.output_dir ./eval_workflow/eval_output/eval_output_8b

2025-10-15 15:37:24 - INFO     - nat.eval.evaluate:446 - Starting evaluation run with config file: eval_workflow/configs/config_d.yml
2025-10-15 15:37:47 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for workflow.llm_name with value: nim_llm_8b with type <class 'str'>)
2025-10-15 15:37:47 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for eval.evaluators.trajectory_accuracy.llm_name with value: nim_llm_8b with type <class 'str'>)
2025-10-15 15:37:47 - INFO     - nat.cli.cli_utils.config_override:105 - Successfully set override for eval.general.output_dir with value: ./eval_workflow/eval_output/eval_output_8b with type <class 'str'>)
2025-10-15 15:37:47 - INFO     - nat.cli.cli_utils.config_override:211 - 

Configuration after overrides:

eval:
  evaluators:
    trajectory_accuracy:
      _type: trajectory
      llm_name: nim_llm_8b
  general:
    dataset:
      _type: json
      file_path: ./eval_workflow/data/eval_data.j

2025-10-15 15:37:47 - INFO     - phoenix.config:1521 - 📋 Ensuring phoenix working directory: /Users/bbednarski/.phoenix
2025-10-15 15:37:47 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_b8878efa-d67f-4038-930e-14b833795fb2 initialized


Running workflow:   0%|          | 0/20 [00:00<?, ?it/s]

2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:37:59 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 15:38:01 - INFO     - nat.agent.react_agent.agent:169 - 
------------------------------
[AGENT]
Agent input: Search for information about the latest NASA Mars mission and summarize 

Running workflow:   5%|▌         | 1/20 [00:03<00:58,  3.09s/it]

...

2025-10-15 15:50:40 - ERROR    - nat.eval.trajectory_evaluator.evaluate:68 - Error evaluating trajectory for question: What are the main differences between Python 3.11 and Python 3.12? Search for official documentation., Error: [504] Gateway Timeout
{'_content': b'', '_content_consumed': True, '_next': None, 'status_code': 504, 'headers': {'Date': 'Wed, 15 Oct 2025 22:50:40 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': '684a123b-86e9-4744-b64f-ee80f3fcef40', 'Nvcf-Status': 'errored', 'Vary': 'Origin'}, 'raw': <urllib3.response.HTTPResponse object at 0x13288d6c0>, 'url': 'https://integrate.api.nvidia.com/v1/chat/completions', 'encoding': None, 'history': [], 'reason': 'Gateway Timeout', 'cookies': <RequestsCookieJar[]>, 'elapsed': datetime.timedelta(seconds=302, microseconds=348183), 'request': <PreparedRequest [POST]>, 'connection': <requests.adapters.HTTPAdapter object at 0x132b9c290>}
Traceback (most recent call

Evaluating Trajectory:  95%|█████████▌| 19/20 [07:17<00:27, 27.69s/it]

2025-10-15 15:52:26 - ERROR    - nat.eval.trajectory_evaluator.evaluate:68 - Error evaluating trajectory for question: How many days are there between January 15, 2024 and March 30, 2024?, Error: Could not find score in model eval output: Since the agent trajectory is empty, let's assume the AI model didn't use any tools to answer the question.

In this case, the AI model's final answer is:
[RESPONSE]
There are 69 days between January 15, 2024 and March 30, 2024.
[END_RESPONSE]

Let's evaluate the answer step by step:

i. Is the final answer helpful?
Yes, the final answer is helpful as it provides the exact number of days between the two dates.

ii. Does the AI language use a logical sequence of tools to answer the question?
No, the AI model didn't use any tools to answer the question, so it's hard to evaluate the sequence of tools.

iii. Does the AI language model use the tools in a helpful way?
No, the AI model didn't use any tools, so it's not possible to evaluate how helpful the to

Evaluating Trajectory: 100%|██████████| 20/20 [07:19<00:00, 20.43s/it]ng Trajectory: 100%|██████████| 20/20 [07:19<00:00, 22.00s/it]
2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:250 - Profiler is not enabled. Skipping profiling.
2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:335 - Workflow output written to eval_workflow/eval_output/eval_output_8b/workflow_output.json
2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:346 - Evaluation results written to eval_workflow/eval_output/eval_output_8b/trajectory_accuracy_output.json
2025-10-15 15:52:26 - WARNING  - nat.eval.evaluate:358 - Workflow execution was interrupted due to an error. The results may be incomplete. You can re-execute evaluation for incomplete results by running `eval` with the --skip_completed_entries flag.


2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:426 - Waiting for export tasks from 1 local exporters (timeout: 60s)
2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:431 - Export tasks completed for exporter: phoenix
2025-10-15 15:52:26 - INFO     - nat.eval.evaluate:435 - All local export task waiting completed


In [ ]:
%%bash
# 70B model
nat eval --config_file ./eval_workflow/configs/config_d.yml \
  --override workflow.llm_name nim_llm_70b \
  --override eval.evaluators.trajectory_accuracy.llm_name nim_llm_70b \
  --override eval.general.output_dir ./eval_workflow/eval_output/eval_output_70b


In [ ]:
# 405B model
nat eval --config_file ./eval_workflow/configs/config_d.yml \
  --override workflow.llm_name nim_llm_405b \
  --override eval.evaluators.trajectory_accuracy.llm_name nim_llm_405b \
  --override eval.general.output_dir ./eval_workflow/eval_output/eval_output_405b

In [61]:
%%bash
nat run --config_file eval_workflow/src/eval_workflow/configs/config_d.yml \
        --input "What is 15% of 847?"

2025-10-15 16:01:34 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'eval_workflow/src/eval_workflow/configs/config_d.yml'


2025-10-15 16:01:35 - INFO     - phoenix.config:1521 - 📋 Ensuring phoenix working directory: /Users/bbednarski/.phoenix
2025-10-15 16:01:35 - INFO     - phoenix.inferences.inferences:112 - Dataset: phoenix_inferences_52114949-7df2-40e0-9d55-408cc4f2ff62 initialized

Configuration Summary:
--------------------
Workflow Type: react_agent
Number of Functions: 1
Number of Function Groups: 0
Number of LLMs: 3
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0

2025-10-15 16:01:47 - INFO     - nat.observability.exporter_manager:269 - Started exporter 'phoenix'
2025-10-15 16:01:49 - INFO     - nat.agent.react_agent.agent:169 - 
------------------------------
[AGENT]
Agent input: What is 15% of 847?
Agent's thoughts: 
Thought: I need to calculate 15% of 847
Action: calculate 15% of 847
Action Input: 847

------------------------------
2025-10-15 16:01:49 - WARNING  - nat.agent.react_age

2025-10-15 16:02:03 - ERROR    - nat.cli.commands.start:239 - Failed to initialize workflow
Error: [###] [{'type': 'string_too_short', 'loc': ('body', 'messages', 14, 'content'), 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]
{'error': "[{'type': 'string_too_short', 'loc': ('body', 'messages', 14, 'content'), 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]"}


CalledProcessError: Command 'b'nat run --config_file eval_workflow/src/eval_workflow/configs/config_d.yml \\\n        --input "What is 15% of 847?"\n'' returned non-zero exit status 1.

Nice to haves:

- Structured report generation (see triage agent)
- model and prompt selection in the same grid search
- question and answer evaluation dataset?
- library of different system prompts for the agent
- automated system prompt generation?